# BBC Projet - Prédiction Des Protéines Membranaires: collection et préparation des données

- Professeur: Xavier Brochet (<a href="mailto:xavier.brochet@heig-vd.ch">xavier.brochet@heig-vd.ch</a>)
- Assistant: Thibault Schowing (<a href="mailto:thibault.schowing@heig-vd.ch">thibault.schowing@heig-vd.ch</a>)

Date: Printemps 2026

Kilian Froidevaux & Basile Buxtorf

Ce notebook ne fait que la collection et préparation des données pour l'entrainement du modèle.

Récupère des protéines annotées expérimentalement de UniProtKB/Swiss-Prot:
  - Positive class: transmembrane / protéines de la membrane cellulaire
  - Negative class: cytoplasmique / nucléique / protéines solubles

Fichiers en sortie:
  - positive_sequences.fasta   -> protéines membranaires
  - negative_sequences.fasta   -> protéines non-membranaires
  - dataset_labels.csv         -> ID de la protéine + label + metadonnées

## Pourquoi ces mots-clés et cette logique de filtrage ?

Dans le cadre du Projet 2, l’objectif est de prédire si une protéine humaine est localisée ou non au niveau de la membrane cellulaire. Notre stratégie de collecte doit donc privilégier des annotations expérimentales fiables, tout en produisant des classes suffisamment propres pour entraîner un modèle.

Les mots-clés choisis suivent les définitions curées de UniProt : [Transmembrane (KW-0812)](https://www.uniprot.org/keywords/KW-0812), [Cell membrane (KW-1003)](https://www.uniprot.org/keywords/KW-1003), [Cytoplasm (KW-0963)](https://www.uniprot.org/keywords/KW-0963) et [Nucleus (KW-0539)](https://www.uniprot.org/keywords/KW-0539).

La classe positive utilise `KW-0812` et `KW-1003` parce qu’ils capturent les protéines explicitement transmembranaires et celles annotées comme associées à la membrane plasmique. Pour notre objectif, cela fait sens : ce sont des protéines attendues du côté "membrane" de la frontière décisionnelle.

La classe négative utilise `KW-0963` et `KW-0539` parce que ce sont des annotations curées fréquentes pour des protéines intracellulaires solubles. Cela reste un proxy utile pour la classe "non membranaire", même si ce n’est pas le complément logique parfait de "membrane cellulaire".

En pratique, cette logique est défendable pour le projet parce qu’elle combine plusieurs contraintes importantes :

- on reste sur des entrées Swiss-Prot revues manuellement pour limiter le bruit d’annotation ;
- on limite la recherche aux taxons mammifères pour rester aligné avec les données du projet ;
- on évite d’utiliser une définition trop large de "non membrane" qui mélangerait des protéines nucléaires, cytoplasmiques et potentiellement d’autres localisations ;
- on exclut les mots-clés membranaires de la requête négative pour réduire les contaminations croisées.

In [ ]:
import requests
import time
import csv
import os
from pathlib import Path

UNIPROT_API     = "https://rest.uniprot.org/uniprotkb/search"
OUTPUT_DIR      = Path("data/raw")
MAX_PER_CLASS   = 2000
BATCH_SIZE      = 500

# Mammalian organism IDs (NCBI taxonomy)
MAMMAL_TAXONS = ["9606"]
#, "10090", "10116"]   # human, mouse, rat
TAXON_FILTER  = " OR ".join(f"organism_id:{t}" for t in MAMMAL_TAXONS)

# UniProt keyword IDs
# KW-0812 = Transmembrane  |  KW-1003 = Cell membrane
# KW-0963 = Cytoplasm      |  KW-0539 = Nucleus
POSITIVE_KEYWORDS = ["KW-0812", "KW-1003"]   # membrane-associated
NEGATIVE_KEYWORDS = ["KW-0963", "KW-0539"]   # soluble / cytoplasmic / nuclear

def build_query(keywords: list[str], taxon_filter: str) -> str:
    """Build a UniProt query for reviewed mammalian proteins with given keywords."""
    kw_part = " OR ".join(f"keyword:{kw}" for kw in keywords)
    return f"reviewed:true AND ({taxon_filter}) AND ({kw_part})"


def fetch_uniprot(query: str, max_results: int) -> list[dict]:
    """
    Fetch protein entries from UniProtKB via the REST API.
    Returns a list of dicts with accession, protein name, gene name,
    organism, sequence, and keywords.
    """
    entries = []
    cursor  = None

    print(f"\n  Query: {query}")
    print(f"  Fetching up to {max_results} entries …\n")

    while len(entries) < max_results:
        params = {
            "query":  query,
            "format": "json",
            "size":   min(BATCH_SIZE, max_results - len(entries)),
        }
        if cursor:
            params["cursor"] = cursor

        response = requests.get(UNIPROT_API, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = data.get("results", [])
        if not results:
            break

        for r in results:
            # Accession
            accession = r.get("primaryAccession", "")

            # Protein name (full name preferred)
            pn = r.get("proteinDescription", {})
            recommended = pn.get("recommendedName", {})
            full_name = recommended.get("fullName", {}).get("value", "")
            if not full_name:
                # fall back to submitted name
                submitted = pn.get("submissionNames", [{}])
                full_name = submitted[0].get("fullName", {}).get("value", "") if submitted else ""

            # Gene name
            genes = r.get("genes", [])
            gene_name = genes[0].get("geneName", {}).get("value", "") if genes else ""

            # Organism
            organism = r.get("organism", {}).get("scientificName", "")

            # Sequence
            sequence = r.get("sequence", {}).get("value", "")

            # Keywords (list of names)
            keywords = [kw.get("name", "") for kw in r.get("keywords", [])]

            if accession and sequence:
                entries.append({
                    "accession": accession,
                    "protein_name": full_name,
                    "gene_name": gene_name,
                    "organism": organism,
                    "sequence": sequence,
                    "keywords": "; ".join(keywords),
                })

        print(f"  Retrieved so far: {len(entries)}")

        # Pagination: check for next cursor in Link header
        link_header = response.headers.get("Link", "")
        if 'rel="next"' in link_header:
            # extract cursor value from URL in Link header
            import re
            match = re.search(r'cursor=([^&>]+)', link_header)
            cursor = match.group(1) if match else None
        else:
            cursor = None

        if not cursor:
            break

        time.sleep(0.3)   # be polite to the API

    return entries


def write_fasta(entries: list[dict], filepath: Path, label: str) -> None:
    """Write entries to a FASTA file."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w") as f:
        for e in entries:
            header = (
                f">{e['accession']} | {e['protein_name']} "
                f"| {e['gene_name']} | {e['organism']} | label={label}"
            )
            f.write(header + "\n")
            # wrap sequence at 60 characters (standard FASTA)
            seq = e["sequence"]
            for i in range(0, len(seq), 60):
                f.write(seq[i:i+60] + "\n")
    print(f"  FASTA written → {filepath}  ({len(entries)} sequences)")


def write_csv_labels(positive: list[dict], negative: list[dict], filepath: Path) -> None:
    """Write a CSV with accession, label, and metadata for both classes."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["accession", "label", "protein_name", "gene_name", "organism", "seq_length", "keywords"]
    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for e in positive:
            writer.writerow({
                "accession":    e["accession"],
                "label":        1,
                "protein_name": e["protein_name"],
                "gene_name":    e["gene_name"],
                "organism":     e["organism"],
                "seq_length":   len(e["sequence"]),
                "keywords":     e["keywords"],
            })
        for e in negative:
            writer.writerow({
                "accession":    e["accession"],
                "label":        0,
                "protein_name": e["protein_name"],
                "gene_name":    e["gene_name"],
                "organism":     e["organism"],
                "seq_length":   len(e["sequence"]),
                "keywords":     e["keywords"],
            })
    total = len(positive) + len(negative)
    print(f"  CSV written    → {filepath}  ({total} rows: {len(positive)} pos, {len(negative)} neg)")


def remove_duplicates(entries: list[dict]) -> list[dict]:
    """Remove entries with duplicate accessions."""
    seen = set()
    unique = []
    for e in entries:
        if e["accession"] not in seen:
            seen.add(e["accession"])
            unique.append(e)
    return unique


def check_cross_contamination(positive: list[dict], negative: list[dict]) -> tuple[list, list]:
    """
    Remove any accession that appears in both classes.
    This can happen if a protein has both membrane and cytoplasmic annotations.
    """
    pos_ids = {e["accession"] for e in positive}
    neg_ids = {e["accession"] for e in negative}
    overlap = pos_ids & neg_ids

    if overlap:
        print(f"\n  ⚠  {len(overlap)} accessions found in both classes — removing from negative set.")
        negative = [e for e in negative if e["accession"] not in overlap]

    return positive, negative


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("  UniProt Data Collection — Membrane Protein Prediction")
    print("=" * 60)

    # ── Positive class: membrane proteins ──
    print("\n[1/4] Fetching POSITIVE class (membrane proteins) …")
    pos_query   = build_query(POSITIVE_KEYWORDS, TAXON_FILTER)
    positive    = fetch_uniprot(pos_query, MAX_PER_CLASS)
    positive    = remove_duplicates(positive)
    print(f"  Unique positive entries: {len(positive)}")

    # ── Negative class: cytoplasmic / nuclear proteins ──
    print("\n[2/4] Fetching NEGATIVE class (cytoplasmic / nuclear proteins) …")
    neg_query   = build_query(NEGATIVE_KEYWORDS, TAXON_FILTER)

    # Exclude any protein that also has a membrane keyword to keep classes clean
    membrane_exclusion = " NOT (keyword:KW-0812 OR keyword:KW-1003)"
    neg_query  += membrane_exclusion

    negative    = fetch_uniprot(neg_query, MAX_PER_CLASS)
    negative    = remove_duplicates(negative)
    print(f"  Unique negative entries: {len(negative)}")

    # ── Cross-contamination check ──
    print("\n[3/4] Checking for cross-contamination …")
    positive, negative = check_cross_contamination(positive, negative)

    # ── Write outputs ──
    print("\n[4/4] Writing output files …")
    write_fasta(positive, OUTPUT_DIR / "positive_sequences.fasta", label="membrane")
    write_fasta(negative, OUTPUT_DIR / "negative_sequences.fasta", label="non-membrane")
    write_csv_labels(positive, negative, OUTPUT_DIR / "dataset_labels.csv")

    print("\n" + "=" * 60)
    print("  Done! Next step: run CD-HIT to reduce sequence redundancy.")
    print("  cd-hit -i data/raw/positive_sequences.fasta \\")
    print("         -o data/processed/positive_nr.fasta -c 0.4 -n 2")
    print("=" * 60)


if __name__ == "__main__":
    main()

  UniProt Data Collection — Membrane Protein Prediction

[1/4] Fetching POSITIVE class (membrane proteins) …

  Query: reviewed:true AND (organism_id:9606) AND (keyword:KW-0812 OR keyword:KW-1003)
  Fetching up to 2000 entries …

  Retrieved so far: 500
  Retrieved so far: 1000
  Retrieved so far: 1500
  Retrieved so far: 2000
  Unique positive entries: 2000

[2/4] Fetching NEGATIVE class (cytoplasmic / nuclear proteins) …

  Query: reviewed:true AND (organism_id:9606) AND (keyword:KW-0963 OR keyword:KW-0539) NOT (keyword:KW-0812 OR keyword:KW-1003)
  Fetching up to 2000 entries …

  Retrieved so far: 500
  Retrieved so far: 1000
  Retrieved so far: 1500
  Retrieved so far: 2000
  Unique negative entries: 2000

[3/4] Checking for cross-contamination …

[4/4] Writing output files …
  FASTA written → data/raw/positive_sequences.fasta  (2000 sequences)
  FASTA written → data/raw/negative_sequences.fasta  (2000 sequences)
  CSV written    → data/raw/dataset_labels.csv  (4000 rows: 2000 pos

> Note: To use `cdhit`
```
conda create -n cdhit cd-hit -c bioconda -c conda-forge
conda activate cdhit
```

cd-hit clusterise redondant sequence to 1 if at least 40% of the sequence is the same. This is to have a highely diversed dataset as much as possible.